In [1]:
!pip install sentence-transformers
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... - \ | / done
  Getting requirements to build wheel ... - done
  Preparing metadata (pyproject.toml) ... - done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.7/283.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 k

In [2]:
!mkdir base-chroma
!mkdir finetuned-chroma

In [3]:
from typing import Literal, List
import numpy as np

from sentence_transformers import SentenceTransformer

# experiment with "BAAI/bge-large-en-v1.5" & "BAAI/bge-base-en-v1.5" later
EMBED_MODEL = Literal["BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5", "BAAI/bge-large-en-v1.5"]


def sentence_embed(
  texts: str | List[str], model_name_or_path: EMBED_MODEL = "BAAI/bge-small-en-v1.5", device: str = "cpu"
) -> list[list[float]]:
  """
    Embeds the given texts using the specified model.

    Args:
        texts (str | List[str], str]): The list of texts or text to embed.
        model (EMBED_MODEL): The embedding model to use.

    Returns:
        np.ndarray: The embeddings of the texts.
    """
  model = SentenceTransformer(model_name_or_path)
  embeddings: np.ndarray = model.encode(sentences=texts, device=device, show_progress_bar=True)
  embeddings_list: list = embeddings.tolist()
  return embeddings_list

## Ingest with Base BAAI/bge-small-en-v1.5 model

In [4]:
import chromadb
from chromadb import Collection, QueryResult
from chromadb.api import ClientAPI
from pandas import DataFrame

chroma_collection = 'bge_base_astra_collection'
chroma_dir = "/kaggle/working/bge-base-chroma"

chroma_client: ClientAPI = chromadb.PersistentClient(path=chroma_dir)
chroma_collection: Collection = chroma_client.get_or_create_collection(
    name=chroma_collection, metadata={"hnsw:space": "cosine"}
)


def ingest(
    data: DataFrame,
    doc_col: str,
    id_col: str | None,
    meta_col: list[str] | None = None,
    model_name_or_path: EMBED_MODEL = "BAAI/bge-small-en-v1.5"
) -> None:
    # Create a list of list of floats with the em
    _docs: list[str] = data[doc_col].tolist()

    # Create a list of str with the id column
    if id_col:
        _ids: list[str] = data[id_col].tolist()
    else:
        _ids = [str(i) for i in range(len(data))]

    # Create a list of dictionaries with the metadata columns
    if meta_col:
        _metas: list[dict[str, Any]] | None = data[meta_col].to_dict(orient="records")  # type: ignore
    else:
        _metas = None

    # Embed the documents
    _embeds: list[list[float]] = sentence_embed(texts=_docs, model_name_or_path=model_name_or_path, device='cuda')  # type: ignore

    # Ingest the documents
    chroma_collection.add(  # type: ignore
        documents=_docs,
        embeddings=_embeds,  # type: ignore
        metadatas=_metas,  # type: ignore
        ids=_ids,
    )

In [5]:
import pandas as pd
from pandas import DataFrame

# print("[ INFO ] Loading data...")
# data: DataFrame = pd.read_csv("/kaggle/input/sub-chunk-kb-acl-meta-100k/sub_chunk_kb_acl-100k.csv")  # type: ignore
# print("[ INFO ] Data loaded.")

# import numpy as np

# # Split the data into smaller batches
# batch_size = 20000  # Adjust the batch size as needed
# num_batches = int(np.ceil(len(data) / batch_size))

# for i in range(num_batches):
#     start_idx = i * batch_size
#     end_idx = min((i + 1) * batch_size, len(data))
#     batch_data = data.iloc[start_idx:end_idx]
    
#     print(f"[ INFO ] Ingesting data - Batch {i + 1}...")
#     ingest(data=batch_data, doc_col="text", id_col=None, meta_col=["title", "acl_id", "url", "year", "author"], model_name_or_path="BAAI/bge-small-en-v1.5")  # type: ignore
#     print(f"[ INFO ] Batch {i + 1} ingested.")


## bge small finetuned

In [6]:
chroma_collection = 'bge_small_finetuned_astra_collection'
chroma_dir = "/kaggle/working/bge-small-finetuned-chroma"

chroma_client: ClientAPI = chromadb.PersistentClient(path=chroma_dir)
chroma_collection: Collection = chroma_client.get_or_create_collection(
    name=chroma_collection, metadata={"hnsw:space": "cosine"}
)

print("[ INFO ] Loading data...")
data: DataFrame = pd.read_csv("/kaggle/input/sub-chunk-kb-acl-meta-100k/sub_chunk_kb_acl-100k.csv")  # type: ignore
print("[ INFO ] Data loaded.")

import numpy as np

# Split the data into smaller batches
batch_size = 20000  # Adjust the batch size as needed
num_batches = int(np.ceil(len(data) / batch_size))

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(data))
    batch_data = data.iloc[start_idx:end_idx]
    
    print(f"[ INFO ] Ingesting data - Batch {i + 1}...")
    ingest(data=batch_data, doc_col="text", id_col=None, meta_col=["title", "acl_id", "url", "year", "author"], model_name_or_path="/kaggle/input/finetuned-bge-models/bge-small_finetuned")  # type: ignore
    print(f"[ INFO ] Batch {i + 1} ingested.")


[ INFO ] Loading data...
[ INFO ] Data loaded.
[ INFO ] Ingesting data - Batch 1...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 1 ingested.
[ INFO ] Ingesting data - Batch 2...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 2 ingested.
[ INFO ] Ingesting data - Batch 3...


...

[ INFO ] Batch 3 ingested.
[ INFO ] Ingesting data - Batch 4...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 4 ingested.
[ INFO ] Ingesting data - Batch 5...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 5 ingested.


## bge large

In [7]:
chroma_collection = 'bge_large_astra_collection'
chroma_dir = "/kaggle/working/bge-large-chroma"

chroma_client: ClientAPI = chromadb.PersistentClient(path=chroma_dir)
chroma_collection: Collection = chroma_client.get_or_create_collection(
    name=chroma_collection, metadata={"hnsw:space": "cosine"}
)

print("[ INFO ] Loading data...")
data: DataFrame = pd.read_csv("/kaggle/input/sub-chunk-kb-acl-meta-100k/sub_chunk_kb_acl-100k.csv")  # type: ignore
print("[ INFO ] Data loaded.")

import numpy as np

# Split the data into smaller batches
batch_size = 20000  # Adjust the batch size as needed
num_batches = int(np.ceil(len(data) / batch_size))

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(data))
    batch_data = data.iloc[start_idx:end_idx]
    
    print(f"[ INFO ] Ingesting data - Batch {i + 1}...")
    ingest(data=batch_data, doc_col="text", id_col=None, meta_col=["title", "acl_id", "url", "year", "author"], model_name_or_path="BAAI/bge-large-en-v1.5")  # type: ignore
    print(f"[ INFO ] Batch {i + 1} ingested.")


[ INFO ] Loading data...
[ INFO ] Data loaded.
[ INFO ] Ingesting data - Batch 1...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

[ INFO ] Batch 3 ingested.
[ INFO ] Ingesting data - Batch 4...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 4 ingested.
[ INFO ] Ingesting data - Batch 5...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 5 ingested.


### bge-large finetuned

In [8]:
chroma_collection = 'bge_large_finetuned_astra_collection'
chroma_dir = "/kaggle/working/bge-large-finetuned-chroma"

chroma_client: ClientAPI = chromadb.PersistentClient(path=chroma_dir)
chroma_collection: Collection = chroma_client.get_or_create_collection(
    name=chroma_collection, metadata={"hnsw:space": "cosine"}
)

print("[ INFO ] Loading data...")
data: DataFrame = pd.read_csv("/kaggle/input/sub-chunk-kb-acl-meta-100k/sub_chunk_kb_acl-100k.csv")  # type: ignore
print("[ INFO ] Data loaded.")

import numpy as np

# Split the data into smaller batches
batch_size = 20000  # Adjust the batch size as needed
num_batches = int(np.ceil(len(data) / batch_size))

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(data))
    batch_data = data.iloc[start_idx:end_idx]
    
    print(f"[ INFO ] Ingesting data - Batch {i + 1}...")
    ingest(data=batch_data, doc_col="text", id_col=None, meta_col=["title", "acl_id", "url", "year", "author"], model_name_or_path="/kaggle/input/finetuned-bge-models/bge-large_finetuned")  # type: ignore
    print(f"[ INFO ] Batch {i + 1} ingested.")


[ INFO ] Loading data...
[ INFO ] Data loaded.
[ INFO ] Ingesting data - Batch 1...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 1 ingested.
[ INFO ] Ingesting data - Batch 2...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 2 ingested.
[ INFO ] Ingesting data - Batch 3...


...

[ INFO ] Batch 3 ingested.
[ INFO ] Ingesting data - Batch 4...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 4 ingested.
[ INFO ] Ingesting data - Batch 5...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

[ INFO ] Batch 5 ingested.
